# T2 gate — 2-pass FP64 FFT (`codex/t2-two-pass-v1`)

**Runtime → Change runtime type → A100（或 L4 / G4）→ Run all。** 跑完看最後一格的 `VERDICT`，把最後一格整段輸出貼回給 Claude。

1. 環境 + clone branch
2. 編譯 + `fft2_test`（正確性 gate，rel L2 ≤ 1e-12 vs cuFFT）
3. ncu：每次 `execute()` 的 kernel 數、DRAM 流量（2^24 b1、2^22 b8）
4. 自動判定 + 摘要

非 A100-SXM4-40GB（L4、G4…）的結果只算正確性證據；正式效能數字只認 A100-SXM4-40GB。

In [ ]:
import subprocess, os, re, csv, io, json
BRANCH = "codex/t2-two-pass-v1"
REPO = "/content/cfft"
OUT = "/content/t2_gate"
os.makedirs(OUT, exist_ok=True)

def sh(cmd, log=None):
    """Run a bash command, stream output, optionally tee to a log; never raise (Run all keeps going)."""
    p = subprocess.Popen(["bash", "-lc", cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        print(line, end="")
        lines.append(line)
    p.wait()
    if log:
        with open(log, "w") as f:
            f.writelines(lines)
    return p.returncode, "".join(lines)

_, gpu = sh("nvidia-smi --query-gpu=name,compute_cap,memory.total,driver_version --format=csv,noheader")
GPU_NAME = gpu.strip().split(",")[0]
ARCH = "sm_" + gpu.strip().split(",")[1].strip().replace(".", "")
sh("nvcc --version | tail -2; ncu --version | tail -1")
sh(f"rm -rf {REPO} && git clone -q -b {BRANCH} https://github.com/justin3728/cuda-fft-optimization.git {REPO} && cd {REPO} && git log --oneline -1")
_, COMMIT = sh(f"cd {REPO} && git rev-parse --short HEAD")
print("ARCH =", ARCH)

In [ ]:
rc_build, _ = sh(f"cd {REPO} && make -C fft2 CUDA_ARCH={ARCH} 2>&1 | tail -20; test -x fft2/fft2_test", log=f"{OUT}/build.log")
if rc_build == 0:
    sh(f"cd {REPO} && fft2/fft2_test --describe", log=f"{OUT}/describe.log")
    rc_test, test_out = sh(f"cd {REPO} && fft2/fft2_test", log=f"{OUT}/fft2_test.log")
else:
    rc_test, test_out = -1, ""
    print("BUILD FAILED — see build.log")

In [ ]:
DRIVER = r"""
#include "fft2.h"
#include <cstdio>
#include <cstdlib>
int main(int c, char** v) {
  size_t n = size_t{1} << atoi(v[1]); int b = atoi(v[2]);
  fft2::Plan p(n, b);
  cuDoubleComplex *in, *out; size_t bytes = n * b * sizeof(cuDoubleComplex);
  if (cudaMalloc(&in, bytes) || cudaMalloc(&out, bytes)) { printf("alloc failed\n"); return 1; }
  cudaMemset(in, 0, bytes); cudaDeviceSynchronize();
  p.execute(in, out, 0);
  cudaError_t e = cudaDeviceSynchronize();
  printf("sync=%s bytes=%zu\n", cudaGetErrorString(e), bytes);
  return e == cudaSuccess ? 0 : 1;
}
"""
open("/content/one_exec.cu", "w").write(DRIVER)
NCU = {}
_q = subprocess.run(["bash", "-lc", "ncu --query-metrics"], capture_output=True, text=True).stdout.split()
RD, WR = ("dram__bytes_read", "dram__bytes_write") if "dram__bytes_read" in _q else ("dram__bytes_op_read", "dram__bytes_op_write")
CANON = {RD + ".sum": "dram__bytes_read.sum", WR + ".sum": "dram__bytes_write.sum"}
print("DRAM metrics:", RD, WR)
if rc_build == 0:
    sh(f"cd {REPO} && nvcc -O3 -std=c++17 -arch={ARCH} -Ifft2 /content/one_exec.cu fft2/fft2.cu -o /content/one_exec")
    for lg, b in [(24, 1), (22, 8)]:
        tag = f"N2^{lg}_b{b}"
        rc, out = sh(f"ncu --csv --clock-control none --metrics {RD}.sum,{WR}.sum,gpu__time_duration.sum "
                     f"/content/one_exec {lg} {b}", log=f"{OUT}/ncu_{tag}.csv")
        rows = [r for r in csv.reader(io.StringIO(out)) if len(r) > 10]
        if not rows or "Kernel Name" not in rows[0]:
            NCU[tag] = {"error": "ncu produced no metrics (permission? see log)"}
            continue
        hdr = rows[0]; K, M, U, V = (hdr.index(x) for x in ("Kernel Name", "Metric Name", "Metric Unit", "Metric Value"))
        ID = hdr.index("ID")
        kernels = {}
        for r in rows[1:]:
            k = kernels.setdefault(r[ID], {"name": re.search(r"(pass1_columns|pass2_rows)", r[K]).group(1) if re.search(r"(pass1_columns|pass2_rows)", r[K]) else r[K][:40]})
            try:
                val = float(r[V].replace(",", ""))
            except ValueError:  # "n/a": metric unavailable on this GPU
                continue
            unit = r[U].lower()
            scale = {"byte": 1, "kbyte": 1e3, "mbyte": 1e6, "gbyte": 1e9, "ns": 1, "us": 1e3, "usecond": 1e3, "nsecond": 1, "ms": 1e6, "msecond": 1e6}.get(unit, 1)
            k[CANON.get(r[M], r[M])] = val * scale
        bytes_ = (1 << lg) * b * 16
        NCU[tag] = {"bytes": bytes_, "kernels": list(kernels.values())}
print(json.dumps(NCU, indent=1))

In [ ]:
print("=" * 70)
print(f"GPU={GPU_NAME}  ARCH={ARCH}  commit={COMMIT.strip()}")
ok = True
summary = [l for l in test_out.splitlines() if l.startswith("SUMMARY")]
fails = [l for l in test_out.splitlines() if " FAIL" in l]
print("build:", "OK" if rc_build == 0 else "FAILED")
print("fft2_test exit:", rc_test)
print(summary[-1] if summary else "SUMMARY missing")
m = re.search(r"passed=(\d+) failed=(\d+) skipped=(\d+)", summary[-1] if summary else "")
if rc_build != 0 or rc_test != 0 or not m or int(m.group(2)) != 0 or "api=PASS" not in (summary[-1] if summary else ""):
    ok = False
for l in fails: print("  FAIL ->", l)
skips = [l for l in test_out.splitlines() if "SKIP" in l]
for l in skips: print("  skip ->", l.split(" SKIP")[0])
errs = [float(x) for x in re.findall(r"rel_l2=([0-9.eE+-]+)", test_out)]
if errs: print(f"rel_l2 range: {min(errs):.2e} .. {max(errs):.2e}  (gate 1e-12)")

print("-" * 70)
for tag, d in NCU.items():
    if "error" in d:
        print(tag, "ncu:", d["error"]); continue
    names = [k["name"] for k in d["kernels"]]
    two = names == ["pass1_columns", "pass2_rows"]
    print(f"{tag}: kernels={names}  {'OK (2 launches)' if two else 'UNEXPECTED'}")
    ok &= two
    for k in d["kernels"]:
        rd, wr = k.get("dram__bytes_read.sum", 0), k.get("dram__bytes_write.sum", 0)
        if not (rd and wr): print(f"   {k['name']:<14} DRAM bytes n/a on this GPU")
        t = k.get("gpu__time_duration.sum", 0)
        print(f"   {k['name']:<14} read={rd/d['bytes']:.2f}x  write={wr/d['bytes']:.2f}x  (1.00x = one pass; DRAM calib: L4 ~1.15x, A100 not yet measured)  "
              f"time={t/1e3:.1f} us  eff_BW={(rd+wr)/t if t else 0:.0f} GB/s")
print("=" * 70)
print("VERDICT:", "PASS" if ok else "FAIL", "" if "A100-SXM4-40GB" in GPU_NAME else "| not A100-SXM4-40GB: correctness evidence only, timings are not results")
sh(f"cd /content && tar czf t2_gate_{ARCH}.tgz t2_gate && ls -la t2_gate_{ARCH}.tgz")